# Step 3 pilot - A/B/C Stockfish-information conditions

This notebook generates one explanation per condition (A, B, and C), position, and model for the fixed 10-position pilot. The three calls use independent chat histories. Condition A receives only the position and played move; B also receives Stockfish evaluations before and after the move; C receives the same evaluations plus the best principal variation from the original position. Ollama runs locally; no hosted LLM API is used.

## Installed local models

These tags, short IDs, and sizes are from the `ollama list` output supplied for this project. The short IDs are pinned in `config.py`; the notebook verifies them before generation. Each per-model manifest records the full Ollama model digest and server version.

| Model tag | Local ID prefix | Listed size |
|---|---|---:|
| `qwen2.5:7b` | `845dbda0ea48` | 4.7 GB |
| `deepseek-r1:8b` | `6995872bfe4c` | 5.2 GB |
| `phi4-mini:latest` | `78fad5d182a7` | 2.5 GB |
| `llama3.1:8b` | `46e0c10c039e` | 4.9 GB |

## Reproducibility settings

- Project Python: 3.11.9, using the `.venv` kernel and `requirements-lock.txt`. Ollama is installed separately from Python packages.
- Four models x 10 positions x 3 conditions x one call per condition = 120 generations by default (30 per model). The main-run `LLM_REPEATS` setting does not apply to this pilot. Each model gets its own output folder.
- All conditions use the same system prompt: `You are a strong chess player and coach.` A receives the FEN, side to move, and played SAN move. B adds pre- and post-move evaluations in White-perspective centipawns. C adds the same evaluations and `engine_best_pv_san`, the best PV from the original position. No label, CP-loss target, or engine-best-move field is sent. The optional ASCII board is off by default.
- Temperature 0.0, seed 46, CPU-only (`num_gpu=0`), context 2048 and output limit 400 tokens. `deepseek-r1:8b` uses context 4096 and output limit 3000 tokens; any `<think>` text is stored separately from the answer.
- The local CPU thread count, Python/platform, package versions, Ollama version, full model digest, input/prompt/settings hashes, notebook-source hash, and Git state are recorded in the per-model manifest.

The notebook verifies the pilot sample, private key, Stockfish result CSV, and both manifests/hashes before joining evidence. The key and labels stay local and are used only for validation/joining. Start Ollama before preflight (`ollama serve`). Keep `RUN_PILOT = False` for **Run All**. When ready, set it to `True` in the run-control cell, execute that cell, then execute only the generation cell. After generation and summaries, the optional reproducibility cell is also off by default; set `RUN_REPRO_CHECK = True` and execute only that cell to repeat one or two positions across all models and conditions. It writes no result files. Its displayed table is notebook output, so clear the output before saving if you don't want it retained. The mock mode is an offline smoke test, not a model result.

In [1]:
import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import re
import statistics
import subprocess
import sys
import urllib.error
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

import config

ROOT = config.ROOT
MONTH = config.MONTH
PILOT_DIR = config.OUT_DIR / "step1_pilot" / MONTH
PILOT_CSV = PILOT_DIR / "pilot_for_llm.csv"
PILOT_KEY_PATH = PILOT_DIR / "pilot_key.csv"
PILOT_MANIFEST_PATH = PILOT_DIR / "pilot_manifest.json"
STOCKFISH_DIR = config.OUT_DIR / "stockfish_recheck" / MONTH
STOCKFISH_CSV = STOCKFISH_DIR / "positions_stockfish.csv"
STOCKFISH_MANIFEST_PATH = STOCKFISH_DIR / "stockfish_manifest.json"
NOTEBOOK_PATH = ROOT / "step3_llm_pilot.ipynb"
MODELS = list(config.LLM_PILOT_MODELS)
CONDITIONS = list(config.LLM_PILOT_CONDITIONS)
REPEATS_PER_CONDITION = config.LLM_PILOT_REPEATS_PER_CONDITION
LIMIT = None
NUM_THREADS = max(1, (os.cpu_count() or 2) // 2)
WITH_BOARD = False
USE_MOCK = False
OUTPUT_ROOT = config.OUT_DIR / "step3_llm_pilot" / (MONTH + ("_MOCK" if USE_MOCK else ""))

if tuple(sys.version_info[:3]) != (3, 11, 9):
    raise RuntimeError(f"Use the project's Python 3.11.9 environment; got {sys.version.split()[0]}.")
if config.LLM_BACKEND != "ollama" or config.LLM_DEVICE != "cpu":
    raise ValueError("This pilot is pinned to the local Ollama backend and CPU execution.")
if not MODELS or len(MODELS) != len(set(MODELS)):
    raise ValueError("LLM_PILOT_MODELS must contain unique model tags.")
if set(MODELS) != set(config.LLM_PILOT_MODEL_ID_PREFIXES):
    raise ValueError("Configured model tags and pinned Ollama ID prefixes do not match.")
if CONDITIONS != ["A", "B", "C"] or REPEATS_PER_CONDITION != 1:
    raise ValueError("This pilot is pinned to one independent generation for each of conditions A, B, and C.")
if NUM_THREADS < 1 or (LIMIT is not None and LIMIT < 1):
    raise ValueError("NUM_THREADS and any LIMIT must be positive.")
for required_path in (PILOT_CSV, PILOT_KEY_PATH, PILOT_MANIFEST_PATH, STOCKFISH_CSV, STOCKFISH_MANIFEST_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required Step 1 pilot or Stockfish provenance file is missing: {required_path}")

OLLAMA_URL = config.LLM_PILOT_OLLAMA_URL
SYSTEM = "You are a strong chess player and coach."
USER_TEMPLATE = (
    "Chess position (FEN): {fen}\n{board}"
    "{side} to move.\nThe move played by the human is {move} (standard algebraic notation).\n\n"
    "Explain why a human might choose this move. Describe what it does and what it changes in the position. "
    "Use 3 to 5 sentences and refer to specific pieces and squares."
)
THINK = re.compile(r"<think>(.*?)</think>", re.S)

print(f"Python: {sys.version.split()[0]} | executable: {sys.executable}")
print(f"pandas: {pd.__version__} | Ollama URL: {OLLAMA_URL}")
print(f"Input: {PILOT_CSV} | models: {MODELS} | conditions: {CONDITIONS} | calls/condition: {REPEATS_PER_CONDITION} | CPU threads: {NUM_THREADS}")

Python: 3.11.9 | executable: d:\files for Germany\srh\GitHub files\chess-explanations-thesis\.venv\Scripts\python.exe
pandas: 3.0.1 | Ollama URL: http://localhost:11434
Input: D:\files for Germany\srh\GitHub files\chess-explanations-thesis\outputs\step1_pilot\2026-10\pilot_for_llm.csv | models: ['qwen2.5:7b', 'deepseek-r1:8b', 'phi4-mini:latest', 'llama3.1:8b'] | conditions: ['A', 'B', 'C'] | calls/condition: 1 | CPU threads: 8


In [2]:
def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while chunk := stream.read(1 << 24):
            digest.update(chunk)
    return digest.hexdigest()


def hash_named_files(directory, names):
    digest = hashlib.sha256()
    for name in names:
        path = directory / name
        if not path.is_file():
            raise FileNotFoundError(f"Manifest output is missing: {path}")
        digest.update(name.encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def notebook_source_sha256():
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    source = "\n".join(
        "".join(cell["source"])
        for cell in notebook["cells"]
        if "step3-source" in cell.get("metadata", {}).get("tags", [])
    )
    return sha256_bytes(source.encode("utf-8"))


def git(*args):
    try:
        result = subprocess.run(
            ["git", "-C", str(ROOT), *args],
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


def build_prompt(row, condition, evidence):
    board_text = ""
    if WITH_BOARD:
        import chess
        board_text = "Board (uppercase = White, lowercase = Black):\n" + str(chess.Board(row["fen"])) + "\n"
    prompt = USER_TEMPLATE.format(
        fen=row["fen"],
        board=board_text,
        side=row["side_to_move"].capitalize(),
        move=row["move_to_explain_san"],
    )
    if condition in {"B", "C"}:
        prompt += (
            f"\nStockfish evaluation before the candidate move (White perspective; positive favors White): "
            f"{evidence['eval_before_cp']:+d} cp.\n"
            f"Stockfish evaluation after the candidate move (White perspective; positive favors White): "
            f"{evidence['eval_after_cp']:+d} cp.\n"
        )
    if condition == "C":
        prompt += "Stockfish principal variation from the original position (SAN): "
        prompt += ", ".join(evidence["root_pv_san"]) + ".\n"
    return prompt


def split_thinking(content, thinking_field=""):
    thinking = thinking_field or ""
    found = THINK.findall(content)
    text = THINK.sub("", content)
    thinking += "\n".join(found)
    if "</think>" in text:
        head, text = text.split("</think>", 1)
        thinking += head
    unterminated = "<think>" in text
    if unterminated:
        thinking += text.split("<think>", 1)[1]
        text = text.split("<think>", 1)[0]
    return text.strip(), thinking.strip(), unterminated


def write_json_atomic(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


class OllamaBackend:
    def _call(self, path, payload=None):
        request = urllib.request.Request(
            OLLAMA_URL + path,
            data=None if payload is None else json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
        )
        try:
            with urllib.request.urlopen(request, timeout=1800) as response:
                return json.loads(response.read())
        except urllib.error.URLError as error:
            raise RuntimeError(
                f"Cannot reach Ollama at {OLLAMA_URL}; start it with `ollama serve` and pull the configured models."
            ) from error

    def version(self):
        version = self._call("/api/version").get("version")
        if not version:
            raise RuntimeError("Ollama /api/version did not return a version.")
        return version

    def models(self):
        return self._call("/api/tags").get("models", [])

    def digest(self, model):
        for item in self.models():
            if item.get("name") == model or item.get("model") == model:
                return item.get("digest")
        raise RuntimeError(f"Model {model!r} is not installed with that exact tag; check `ollama list`.")

    def chat(self, model, system, user, options):
        response = self._call(
            "/api/chat",
            {
                "model": model,
                "stream": False,
                "keep_alive": "15m",
                "options": options,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            },
        )
        message = response.get("message", {})
        return {
            "content": message.get("content", ""),
            "thinking": message.get("thinking", ""),
            "done_reason": response.get("done_reason"),
            "eval_count": response.get("eval_count"),
            "prompt_eval_count": response.get("prompt_eval_count"),
            "eval_s": (response.get("eval_duration") or 0) / 1e9,
            "total_s": (response.get("total_duration") or 0) / 1e9,
        }

    def warmup(self, model, options):
        self.chat(model, "Reply with one word.", "Say OK.", {**options, "num_predict": 1})


class MockBackend:
    def __init__(self):
        self.calls = 0

    def version(self):
        return "mock"

    def digest(self, model):
        return "mock-" + sha256_bytes(model.encode("utf-8"))[:12]

    def warmup(self, model, options):
        return None

    def chat(self, model, system, user, options):
        self.calls += 1
        move = re.search(r"move played by the human is (\S+)", user).group(1)
        value = int(sha256_bytes((model + user).encode("utf-8"))[:6], 16)
        content = f"The move {move} develops the position and improves piece activity. It is a reasonable choice. [{value % 7}]"
        if "phi" in model and self.calls % 2 == 0:
            content += " Variant."
        done_reason = "stop"
        if "deepseek" in model:
            content = f"<think>Let me look at {move} carefully.</think>" + content
            if value % 5 == 0:
                content, done_reason = "<think>I am still thinking about", "length"
        elif "llama" in model and value % 4 == 0:
            content, done_reason = content[:40], "length"
        return {
            "content": content,
            "thinking": "",
            "done_reason": done_reason,
            "eval_count": 60,
            "prompt_eval_count": 120,
            "eval_s": 6.0 + value % 3,
            "total_s": 7.0 + value % 3,
        }


def read_done(path, settings_sha256):
    completed = set()
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                record = json.loads(line)
                if record.get("settings_sha256") != settings_sha256:
                    raise RuntimeError(
                        f"{path} was made with different model/input/settings/source. "
                        "Preserve the old folder before starting a new run."
                    )
                if "condition" not in record:
                    raise RuntimeError(f"{path} contains records from the previous single-condition format.")
                completed.add((record["pilot_id"], record["condition"], record["repeat"]))
    return completed


def model_folder_name(model):
    return re.sub(r"[^A-Za-z0-9._-]", "_", model)

In [3]:
pilot_manifest = json.loads(PILOT_MANIFEST_PATH.read_text(encoding="utf-8"))
stockfish_manifest = json.loads(STOCKFISH_MANIFEST_PATH.read_text(encoding="utf-8"))
pilot_rows = list(csv.DictReader(PILOT_CSV.open(encoding="utf-8", newline="")))
expected_input_columns = ["pilot_id", "fen", "side_to_move", "move_to_explain_san"]
expected_columns = set(expected_input_columns)
if not pilot_rows or set(pilot_rows[0]) != expected_columns:
    raise ValueError("pilot_for_llm.csv must contain only pilot_id, fen, side_to_move, and move_to_explain_san.")
if pilot_manifest.get("step") != "step1_pilot_sample" or pilot_manifest.get("month") != MONTH:
    raise ValueError("Pilot manifest has an unexpected step or month.")
if pilot_manifest.get("positions_per_kind") != config.PILOT_POSITIONS_PER_KIND:
    raise ValueError("Pilot manifest sample size differs from config.py.")
if pilot_manifest.get("llm_input_columns") != expected_input_columns:
    raise ValueError("Pilot manifest input-column list does not match pilot_for_llm.csv.")
if len(pilot_rows) != 2 * config.PILOT_POSITIONS_PER_KIND:
    raise ValueError("Pilot CSV row count differs from the configured balanced sample.")
if sha256_file(PILOT_CSV) != pilot_manifest.get("llm_input_sha256"):
    raise ValueError("Pilot CSV hash does not match pilot_manifest.json.")
if sha256_file(PILOT_KEY_PATH) != pilot_manifest.get("pilot_key_sha256"):
    raise ValueError("Pilot key hash does not match pilot_manifest.json.")
if sha256_file(STOCKFISH_MANIFEST_PATH) != pilot_manifest.get("source_stockfish_manifest_sha256"):
    raise ValueError("Stockfish manifest changed since the Step 1 pilot sample was created.")
if stockfish_manifest.get("results_sha256") != pilot_manifest.get("source_stockfish_results_sha256"):
    raise ValueError("Pilot manifest points to different Stockfish results.")
if (stockfish_manifest.get("step") != "step2_stockfish_recheck"
        or stockfish_manifest.get("month") != MONTH
        or str(stockfish_manifest.get("stockfish_version")) != str(config.STOCKFISH_VERSION)
        or stockfish_manifest.get("depth") != config.STOCKFISH_DEPTH
        or stockfish_manifest.get("threads") != config.STOCKFISH_THREADS
        or stockfish_manifest.get("hash_mb") != config.STOCKFISH_HASH_MB
        or stockfish_manifest.get("multipv") != config.STOCKFISH_MULTIPV):
    raise ValueError("Stockfish manifest version/settings do not match config.py.")
if "positions_stockfish.csv" not in stockfish_manifest.get("hashed_outputs", []):
    raise ValueError("Stockfish manifest does not include positions_stockfish.csv.")
if hash_named_files(STOCKFISH_DIR, stockfish_manifest["hashed_outputs"]) != stockfish_manifest.get("results_sha256"):
    raise ValueError("Stockfish output files do not match stockfish_manifest.json.")
if sha256_file(STOCKFISH_CSV) != pilot_manifest.get("source_positions_sha256"):
    raise ValueError("Stockfish positions CSV changed since the Step 1 pilot sample was created.")
if pilot_manifest.get("counts") != {
    "good": config.PILOT_POSITIONS_PER_KIND,
    "bad": config.PILOT_POSITIONS_PER_KIND,
}:
    raise ValueError("Pilot manifest does not describe the configured good/bad balance.")
if pilot_manifest.get("labels_in_llm_input") is not False:
    raise ValueError("Pilot manifest does not confirm that labels were excluded from the LLM input.")

pilot_key = pd.read_csv(PILOT_KEY_PATH, dtype={"pilot_id": "string", "position_id": "string"})
stockfish_rows = pd.read_csv(STOCKFISH_CSV, dtype={"position_id": "string"})
key_columns = {"pilot_id", "position_id", "kind", "stockfish_label"}
stockfish_columns = {"position_id", "kind", "stockfish_label", "fen", "move_san", "eval_before_cp", "eval_after_cp", config.LLM_PILOT_PV_COLUMN}
if not key_columns.issubset(pilot_key.columns) or not stockfish_columns.issubset(stockfish_rows.columns):
    raise ValueError("Pilot key or Stockfish CSV is missing columns required for evidence joins.")
if pilot_key["pilot_id"].duplicated().any() or stockfish_rows["position_id"].duplicated().any():
    raise ValueError("Pilot IDs and Stockfish position IDs must be unique for an unambiguous join.")
key_by_pilot = {str(row["pilot_id"]): row for row in pilot_key.to_dict("records")}
stockfish_by_position = {str(row["position_id"]): row for row in stockfish_rows.to_dict("records")}
if set(key_by_pilot) != {row["pilot_id"] for row in pilot_rows}:
    raise ValueError("Pilot key IDs do not match pilot_for_llm.csv.")

def score_cp(value, field, pilot_id):
    if pd.isna(value) or not float(value).is_integer():
        raise ValueError(f"{pilot_id}: Stockfish {field} must be a non-missing integer centipawn score.")
    return int(value)

evidence_by_pilot = {}
for row in pilot_rows:
    pilot_id = row["pilot_id"]
    key_row = key_by_pilot[pilot_id]
    source = stockfish_by_position.get(str(key_row["position_id"]))
    if source is None:
        raise ValueError(f"{pilot_id}: source position is missing from positions_stockfish.csv.")
    if str(source["kind"]) != str(key_row["kind"]) or str(source["stockfish_label"]) != str(key_row["stockfish_label"]):
        raise ValueError(f"{pilot_id}: private key labels do not match the Stockfish source row.")
    if str(source["fen"]) != row["fen"] or str(source["move_san"]) != row["move_to_explain_san"]:
        raise ValueError(f"{pilot_id}: pilot position/move does not match its Stockfish source row.")
    try:
        pv = json.loads(source[config.LLM_PILOT_PV_COLUMN])
    except (TypeError, json.JSONDecodeError) as error:
        raise ValueError(f"{pilot_id}: Stockfish PV is not valid JSON.") from error
    if not isinstance(pv, list) or not pv or not all(isinstance(move, str) and move for move in pv):
        raise ValueError(f"{pilot_id}: Stockfish root PV must be a non-empty list of SAN moves.")
    evidence_by_pilot[pilot_id] = {
        "eval_before_cp": score_cp(source["eval_before_cp"], "eval_before_cp", pilot_id),
        "eval_after_cp": score_cp(source["eval_after_cp"], "eval_after_cp", pilot_id),
        "root_pv_san": pv,
    }

if LIMIT is not None:
    pilot_rows = pilot_rows[:LIMIT]
if not pilot_rows:
    raise ValueError("No pilot rows selected for generation.")

INPUT_CSV_SHA256 = sha256_file(PILOT_CSV)
PILOT_MANIFEST_SHA256 = sha256_file(PILOT_MANIFEST_PATH)
PILOT_KEY_SHA256 = sha256_file(PILOT_KEY_PATH)
STOCKFISH_MANIFEST_SHA256 = sha256_file(STOCKFISH_MANIFEST_PATH)
STOCKFISH_RESULTS_SHA256 = stockfish_manifest["results_sha256"]
PROMPTS = {
    (row["pilot_id"], condition): build_prompt(row, condition, evidence_by_pilot[row["pilot_id"]])
    for row in pilot_rows
    for condition in CONDITIONS
}
PROMPT_SHA256 = sha256_bytes(
    json.dumps([[pilot_id, condition, prompt] for (pilot_id, condition), prompt in sorted(PROMPTS.items())]).encode("utf-8")
)
NOTEBOOK_SOURCE_SHA256 = notebook_source_sha256()
if not NOTEBOOK_SOURCE_SHA256:
    raise RuntimeError("Could not hash the saved Step 3 code cells. Save the notebook and rerun setup.")

BACKEND = MockBackend() if USE_MOCK else OllamaBackend()
OLLAMA_VERSION = BACKEND.version()
MODEL_DIGESTS = {}
if USE_MOCK:
    MODEL_DIGESTS = {model: BACKEND.digest(model) for model in MODELS}
    print("Mock mode: skipping Ollama server and model checks.")
else:
    installed_models = BACKEND.models()
    for model in MODELS:
        match = next(
            (item for item in installed_models if item.get("name") == model or item.get("model") == model),
            None,
        )
        if match is None:
            raise RuntimeError(f"{model} is not installed with the configured tag; check `ollama list`.")
        digest = match.get("digest", "")
        digest_hex = digest.removeprefix("sha256:")
        expected_prefix = config.LLM_PILOT_MODEL_ID_PREFIXES[model]
        if not digest_hex.startswith(expected_prefix):
            raise RuntimeError(
                f"{model} has digest {digest!r}, not pinned ID prefix {expected_prefix!r}. "
                "Verify the model and update config only if intentionally changing versions."
            )
        MODEL_DIGESTS[model] = digest
        print(f"Verified {model}: {digest}")
    print(f"Ollama server version: {OLLAMA_VERSION}")

print(f"Pilot input and Stockfish evidence verified: {len(pilot_rows)} positions")
print(f"Input SHA-256: {INPUT_CSV_SHA256} | Stockfish results SHA-256: {STOCKFISH_RESULTS_SHA256}")
print(f"Saved Step 3 source hash: {NOTEBOOK_SOURCE_SHA256}")

Verified qwen2.5:7b: 845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e
Verified deepseek-r1:8b: 6995872bfe4c521a67b32da386cd21d5c6e819b6e0d62f79f64ec83be99f5763
Verified phi4-mini:latest: 78fad5d182a7c33065e153a5f8ba210754207ba9d91973f57dffa7f487363753
Verified llama3.1:8b: 46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e
Ollama server version: 0.40.16
Pilot input and Stockfish evidence verified: 10 positions
Input SHA-256: 8f01bfa37d696961ae756e825e5fdc45593c965dde5fbbe5a326c06dacc3e5ed | Stockfish results SHA-256: fe0249c236075523393cd9b03532678368a7225e09305e4a56fd2f74a68796b8
Saved Step 3 source hash: 656877d4899b4ab1a5a9a748be330bc8374620918c3f9fb5704be79685101ca5


In [8]:
RUN_PILOT = False

In [9]:
if RUN_PILOT:
    if not MODELS:
        raise ValueError("Select at least one model in MODELS.")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for model in MODELS:
        reasoning = model in config.LLM_PILOT_REASONING_MODELS
        options = {
            "temperature": config.LLM_TEMPERATURE,
            "seed": config.SEED_LLM,
            "num_gpu": 0,
            "num_thread": NUM_THREADS,
            "num_ctx": max(config.LLM_NUM_CTX, config.LLM_PILOT_REASONING_NUM_CTX) if reasoning else config.LLM_NUM_CTX,
            "num_predict": config.LLM_PILOT_REASONING_MAX_TOKENS if reasoning else config.LLM_MAX_TOKENS,
        }
        digest = BACKEND.digest(model)
        if digest != MODEL_DIGESTS[model]:
            raise RuntimeError(f"{model} changed after preflight; stop and verify the installed model.")
        expected_prefix = config.LLM_PILOT_MODEL_ID_PREFIXES[model]
        if not USE_MOCK and not digest.removeprefix("sha256:").startswith(expected_prefix):
            raise RuntimeError(f"{model} no longer matches pinned ID prefix {expected_prefix}.")

        settings = {
            "model": model,
            "mock": USE_MOCK,
            "model_digest": digest,
            "conditions": CONDITIONS,
            "repeats_per_condition": REPEATS_PER_CONDITION,
            "pv_column": config.LLM_PILOT_PV_COLUMN,
            "options": options,
            "prompt_sha256": PROMPT_SHA256,
            "input_csv_sha256": INPUT_CSV_SHA256,
            "pilot_manifest_sha256": PILOT_MANIFEST_SHA256,
            "pilot_key_sha256": PILOT_KEY_SHA256,
            "stockfish_manifest_sha256": STOCKFISH_MANIFEST_SHA256,
            "stockfish_results_sha256": STOCKFISH_RESULTS_SHA256,
            "with_board": WITH_BOARD,
            "ollama_version": OLLAMA_VERSION,
            "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "processor": platform.processor(),
        }
        settings_sha256 = sha256_bytes(json.dumps(settings, sort_keys=True).encode("utf-8"))
        model_dir = OUTPUT_ROOT / model_folder_name(model)
        model_dir.mkdir(parents=True, exist_ok=True)
        generations_path = model_dir / "generations.jsonl"
        completed = read_done(generations_path, settings_sha256)
        expected_count = len(pilot_rows) * len(CONDITIONS) * REPEATS_PER_CONDITION
        print(
            f"\n== {model} (digest {digest.removeprefix('sha256:')[:12]}, "
            f"{len(completed)}/{expected_count} condition-position calls already done) ==",
            flush=True,
        )
        BACKEND.warmup(model, options)
        with generations_path.open("a", encoding="utf-8", newline="\n") as stream:
            for condition in CONDITIONS:
                for row in pilot_rows:
                    repeat = 1
                    pair = (row["pilot_id"], condition, repeat)
                    if pair in completed:
                        continue
                    prompt = PROMPTS[(row["pilot_id"], condition)]
                    result = BACKEND.chat(model, SYSTEM, prompt, options)
                    text, thinking, think_unterminated = split_thinking(result["content"], result["thinking"])
                    eval_count = result["eval_count"]
                    tokens_per_second = (
                        eval_count / result["eval_s"]
                        if eval_count is not None and result["eval_s"]
                        else None
                    )
                    record = {
                        "pilot_id": row["pilot_id"],
                        "repeat": repeat,
                        "model": model,
                        "model_digest": digest,
                        "condition": condition,
                        "text": text,
                        "thinking": thinking,
                        "think_unterminated": think_unterminated,
                        "raw_content": result["content"],
                        "done_reason": result["done_reason"],
                        "truncated": result["done_reason"] == "length",
                        "eval_count": eval_count,
                        "prompt_eval_count": result["prompt_eval_count"],
                        "seconds": round(result["total_s"], 2),
                        "tokens_per_s": None if tokens_per_second is None else round(tokens_per_second, 2),
                        "settings_sha256": settings_sha256,
                        "prompt_sha256": sha256_bytes(prompt.encode("utf-8")),
                        "prompt": prompt,
                        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    }
                    stream.write(json.dumps(record, ensure_ascii=False) + "\n")
                    stream.flush()
                    completed.add(pair)
                    print(
                        f"  {record['pilot_id']} {condition}: {record['seconds']:>6.1f}s  "
                        f"{record['eval_count']} tok  {record['done_reason']}",
                        flush=True,
                    )

        manifest = {
            "step": "step3_llm_pilot",
            "model": model,
            "model_digest": digest,
            "ollama_version": OLLAMA_VERSION,
            "conditions": {
                "A": "Position and played move only.",
                "B": "Condition A plus pre- and post-move White-perspective Stockfish evaluations.",
                "C": "Condition B plus engine_best_pv_san, the best PV from the original position.",
            },
            "settings": settings,
            "settings_sha256": settings_sha256,
            "generations_sha256": sha256_file(generations_path),
            "system_prompt": SYSTEM,
            "user_template": USER_TEMPLATE,
            "prompt_sha256": PROMPT_SHA256,
            "input_csv_sha256": INPUT_CSV_SHA256,
            "pilot_manifest_sha256": PILOT_MANIFEST_SHA256,
            "pilot_key_sha256": PILOT_KEY_SHA256,
            "stockfish_manifest_sha256": STOCKFISH_MANIFEST_SHA256,
            "stockfish_results_sha256": STOCKFISH_RESULTS_SHA256,
            "n_positions": len(pilot_rows),
            "n_conditions": len(CONDITIONS),
            "repeats_per_condition": REPEATS_PER_CONDITION,
            "expected_generations": expected_count,
            "completed_generations": len(completed),
            "config": config.config_snapshot(),
            "git_commit": git("rev-parse", "HEAD"),
            "git_uncommitted_changes": bool(git("status", "--porcelain")),
            "notebook_source_sha256": NOTEBOOK_SOURCE_SHA256,
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "processor": platform.processor(),
            "cpu_count": os.cpu_count(),
            "packages": {
                name: importlib.metadata.version(name)
                for name in ("chess", "numpy", "pandas")
            },
            "mock": USE_MOCK,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
        write_json_atomic(model_dir / "run_manifest.json", manifest)
else:
    print("Pilot generation skipped. Set RUN_PILOT = True and execute only this cell when ready.")


== qwen2.5:7b (digest 845dbda0ea48, 0/30 condition-position calls already done) ==
  P001 A:   16.0s  78 tok  stop
  P002 A:   16.9s  91 tok  stop
  P003 A:   17.7s  103 tok  stop
  P004 A:   20.6s  115 tok  stop
  P005 A:   15.0s  86 tok  stop
  P006 A:   16.5s  90 tok  stop
  P007 A:   15.5s  87 tok  stop
  P008 A:   19.7s  101 tok  stop
  P009 A:   17.4s  89 tok  stop
  P010 A:   19.9s  101 tok  stop
  P001 B:   21.3s  102 tok  stop
  P002 B:   19.0s  104 tok  stop
  P003 B:   23.5s  136 tok  stop
  P004 B:   25.3s  141 tok  stop
  P005 B:   21.1s  117 tok  stop
  P006 B:   15.6s  99 tok  stop
  P007 B:   18.8s  112 tok  stop
  P008 B:   23.5s  136 tok  stop
  P009 B:   15.7s  94 tok  stop
  P010 B:   21.8s  124 tok  stop
  P001 C:   21.8s  118 tok  stop
  P002 C:   20.8s  125 tok  stop
  P003 C:   38.1s  225 tok  stop
  P004 C:   26.3s  155 tok  stop
  P005 C:   19.6s  119 tok  stop
  P006 C:   19.3s  121 tok  stop
  P007 C:   15.9s  100 tok  stop
  P008 C:   21.1s  131 tok  stop


In [10]:
if RUN_PILOT:
    records = []
    for model in MODELS:
        generations_path = OUTPUT_ROOT / model_folder_name(model) / "generations.jsonl"
        records.extend(
            json.loads(line)
            for line in generations_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        )

    summary_rows, comparison_rows, review_rows = [], [], []
    for model in MODELS:
        model_records = [record for record in records if record["model"] == model]
        lookup = {}
        for record in model_records:
            key = (record["pilot_id"], record["condition"])
            if key in lookup:
                raise ValueError(f"Duplicate generation for {model}, {key}.")
            lookup[key] = record
        expected_keys = {(row["pilot_id"], condition) for row in pilot_rows for condition in CONDITIONS}
        if set(lookup) != expected_keys:
            raise RuntimeError(f"{model} does not have exactly one result per selected position and condition.")
        for condition in CONDITIONS:
            condition_records = [record for record in model_records if record["condition"] == condition]
            speeds = [record["tokens_per_s"] for record in condition_records if record["tokens_per_s"]]
            summary_rows.append(
                {
                    "model": model,
                    "condition": condition,
                    "generations": len(condition_records),
                    "truncated": sum(record["truncated"] for record in condition_records),
                    "empty_text": sum(not record["text"] for record in condition_records),
                    "mean_seconds": round(statistics.mean(record["seconds"] for record in condition_records), 1),
                    "mean_tokens_per_s": round(statistics.mean(speeds), 2) if speeds else None,
                    "median_words": statistics.median(len(record["text"].split()) for record in condition_records),
                }
            )
        for row in pilot_rows:
            comparison = {"pilot_id": row["pilot_id"], "model": model}
            for condition in CONDITIONS:
                record = lookup[(row["pilot_id"], condition)]
                comparison[f"condition_{condition}"] = record["text"]
                review_rows.append(
                    {
                        "pilot_id": row["pilot_id"],
                        "model": model,
                        "condition": condition,
                        "repeat": record["repeat"],
                        "text": record["text"],
                        "n_claims": "",
                        "n_incorrect": "",
                        "quality_1to5": "",
                        "notes": "",
                    }
                )
            comparison_rows.append(comparison)

    for name, table in (("summary.csv", summary_rows), ("condition_comparison.csv", comparison_rows), ("review_sheet.csv", review_rows)):
        with (OUTPUT_ROOT / name).open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(table[0]))
            writer.writeheader()
            writer.writerows(table)

    print(pd.DataFrame(summary_rows).to_string(index=False))
    print(f"\nFull per-model generations and manifests: {OUTPUT_ROOT}")
else:
    print("No generation results to summarize yet.")

           model condition  generations  truncated  empty_text  mean_seconds  mean_tokens_per_s  median_words
      qwen2.5:7b         A           10          0           0          17.5               6.31          74.0
      qwen2.5:7b         B           10          0           0          20.5               6.07          93.5
      qwen2.5:7b         C           10          0           0          22.6               6.46          99.0
  deepseek-r1:8b         A           10          1           1         139.0               5.65          95.0
  deepseek-r1:8b         B           10          0           0         107.4               5.61         132.0
  deepseek-r1:8b         C           10          0           0         103.0               5.62         132.0
phi4-mini:latest         A           10          0           0          11.3              11.02          82.5
phi4-mini:latest         B           10          0           0          10.5              11.89          96.5
phi4-mini:

In [12]:
RUN_REPRO_CHECK = True
if RUN_REPRO_CHECK:
    # Read-only check; this cell is excluded from the generation-source hash and writes no files.
    REPRO_CHECK_N_POSITIONS = 2  # Set to 1 for 12 calls, or 2 for 24 calls.
    if USE_MOCK:
        raise RuntimeError("Disable USE_MOCK before checking real pilot reproducibility.")
    if REPRO_CHECK_N_POSITIONS not in {1, 2}:
        raise ValueError("REPRO_CHECK_N_POSITIONS must be 1 or 2.")

    reference_by_model = {}
    manifests = {}
    for model in MODELS:
        model_dir = OUTPUT_ROOT / model_folder_name(model)
        manifest = json.loads((model_dir / "run_manifest.json").read_text(encoding="utf-8"))
        if (manifest.get("model_digest") != BACKEND.digest(model)
                or manifest.get("ollama_version") != OLLAMA_VERSION
                or manifest.get("notebook_source_sha256") != NOTEBOOK_SOURCE_SHA256
                or manifest.get("system_prompt") != SYSTEM):
            raise RuntimeError(f"{model}: model, Ollama version, prompt, or source differs from the saved run.")
        manifests[model] = manifest
        reference = {}
        for line in (model_dir / "generations.jsonl").read_text(encoding="utf-8").splitlines():
            if line.strip():
                record = json.loads(line)
                key = (record["pilot_id"], record["condition"])
                if record.get("settings_sha256") != manifest.get("settings_sha256") or key in reference:
                    raise RuntimeError(f"{model}: saved generations have inconsistent settings or duplicate records.")
                reference[key] = record
        reference_by_model[model] = reference

    selected_ids = [pilot_rows[0]["pilot_id"]]
    failure_ids = [
        pilot_id for (pilot_id, condition), record in reference_by_model["deepseek-r1:8b"].items()
        if condition == "A" and (record["truncated"] or not record["text"])
    ]
    second_id = next((pid for pid in failure_ids if pid not in selected_ids), None)
    if second_id is None:
        second_id = next((row["pilot_id"] for row in pilot_rows if row["pilot_id"] not in selected_ids), None)
    if REPRO_CHECK_N_POSITIONS == 2 and second_id is not None:
        selected_ids.append(second_id)

    check_rows = []
    for model in MODELS:
        reasoning = model in config.LLM_PILOT_REASONING_MODELS
        options = {
            "temperature": config.LLM_TEMPERATURE, "seed": config.SEED_LLM, "num_gpu": 0,
            "num_thread": NUM_THREADS,
            "num_ctx": max(config.LLM_NUM_CTX, config.LLM_PILOT_REASONING_NUM_CTX) if reasoning else config.LLM_NUM_CTX,
            "num_predict": config.LLM_PILOT_REASONING_MAX_TOKENS if reasoning else config.LLM_MAX_TOKENS,
        }
        settings = manifests[model]["settings"]
        if (settings.get("options") != options
                or settings.get("prompt_sha256") != PROMPT_SHA256
                or settings.get("input_csv_sha256") != INPUT_CSV_SHA256
                or settings.get("conditions") != CONDITIONS
                or settings.get("repeats_per_condition") != REPEATS_PER_CONDITION):
            raise RuntimeError(f"{model}: current options or prompts differ from the saved run.")
        for pilot_id in selected_ids:
            for condition in CONDITIONS:
                prompt = PROMPTS[(pilot_id, condition)]
                original = reference_by_model[model][(pilot_id, condition)]
                if original.get("prompt_sha256") != sha256_bytes(prompt.encode("utf-8")):
                    raise RuntimeError(f"{model}/{pilot_id}/{condition}: saved prompt differs.")
                repeated = BACKEND.chat(model, SYSTEM, prompt, options)
                text, thinking, _ = split_thinking(repeated["content"], repeated["thinking"])
                check_rows.append({
                    "model": model, "pilot_id": pilot_id, "condition": condition,
                    "text_identical": text == original["text"],
                    "raw_identical": repeated["content"] == original["raw_content"],
                    "original_empty": not bool(original["text"]), "repeat_empty": not bool(text),
                    "original_truncated": original["truncated"],
                    "repeat_truncated": repeated["done_reason"] == "length",
                    "thinking_identical": thinking == original["thinking"],
                })

    print(f"Re-generated {len(check_rows)} responses for {selected_ids}; no files were written.")
    print(pd.DataFrame(check_rows).to_string(index=False))
else:
    print("Reproducibility check skipped. Set RUN_REPRO_CHECK = True and execute this cell manually.")

Re-generated 24 responses for ['P001', 'P008']; no files were written.
           model pilot_id condition  text_identical  raw_identical  original_empty  repeat_empty  original_truncated  repeat_truncated  thinking_identical
      qwen2.5:7b     P001         A           False          False           False         False               False             False                True
      qwen2.5:7b     P001         B            True           True           False         False               False             False                True
      qwen2.5:7b     P001         C           False          False           False         False               False             False                True
      qwen2.5:7b     P008         A           False          False           False         False               False             False                True
      qwen2.5:7b     P008         B           False          False           False         False               False             False                True